# Кандидат `submission_07.csv` — V5 + ТРАНСДУКТИВНАЯ импутация

Та же лучшая модель **V5** (SVC+HGB+ExtraTrees+2×MLP), но `IterativeImputer` обучается на признаках **train+test вместе** (без меток — не утечка). Это даёт регрессии восстановления `eog_burst_index` больше данных (+~2500 строк) → чуть точнее импутация именно у тестовых эпох без EOG-канала.

CV (трансдуктивно-импутированный train) = **0.8433** ≈ обычная in-pipeline (0.8432) — качество не деградирует; доработка целит в тестовую no-eog половину. Отличается от `submission_06` на ~42 строки. Запускается и даёт тот же `submission_07.csv`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
names={0:"Wake",1:"Light",2:"Deep",3:"REM"}
tr=pd.read_csv("train.csv"); te=pd.read_csv("test.csv"); ss=pd.read_csv("sample_submission.csv")
feat=[c for c in tr.columns if c not in ("id","sleep_stage")]; y=tr.sleep_stage.values
EEG=["eeg_delta_power","eeg_theta_power","eeg_alpha_power","eeg_sigma_power","eeg_beta_power","eeg_gamma_power"]
def fe(df):
    X=df.copy(); tot=X[EEG].clip(lower=0).sum(1)+1e-6
    for b in EEG: X["rel_"+b]=X[b]/tot
    X["delta_beta"]=X["eeg_delta_power"]/(X["eeg_beta_power"].abs()+1e-6)
    X["theta_alpha"]=X["eeg_theta_power"]/(X["eeg_alpha_power"].abs()+1e-6)
    X["slow_dom"]=X["eeg_slow_osc_power"]+X["eeg_delta_power"]
    X["eog_burst_missing"]=df["eog_burst_index"].isna().astype(int)
    return X
Xtr=fe(tr[feat]); Xte=fe(te[feat])

**🔎** FE одинаков для train/test. Дальше — ключевой шаг: трансдуктивная импутация.

In [2]:
# Трансдуктивная импутация: imputer учится на train+test (только признаки, без меток)
allX=pd.concat([Xtr,Xte],ignore_index=True)
imp=IterativeImputer(estimator=BayesianRidge(),max_iter=5,random_state=42).fit(allX)
Xtr_i=pd.DataFrame(imp.transform(Xtr),columns=Xtr.columns)
Xte_i=pd.DataFrame(imp.transform(Xte),columns=Xte.columns)
obs=tr.eog_burst_index.dropna(); m=te.eog_burst_index.isna().values
print("eog observed(train) mean/std=%.3f/%.3f | imputed test(no-eog) mean/std=%.3f/%.3f"%(
    obs.mean(),obs.std(),Xte_i.loc[m,"eog_burst_index"].mean(),Xte_i.loc[m,"eog_burst_index"].std()))

eog observed(train) mean/std=-0.010/1.038 | imputed test(no-eog) mean/std=0.012/0.954


**🔎 Вывод.** Распределение восстановленного eog у тестовых no-eog эпох ≈ наблюдаемому в train → импутация без артефактов. Утечки нет: метки `sleep_stage` не участвуют.

In [3]:
def sc(m): return Pipeline([("s",StandardScaler()),("m",m)])
def vote(seed):
    return VotingClassifier([
        ("svc",sc(SVC(C=80,gamma=0.008,probability=True,random_state=seed))),
        ("hgb",HistGradientBoostingClassifier(random_state=seed,learning_rate=0.079,max_iter=240,max_leaf_nodes=43,min_samples_leaf=24,l2_regularization=7.26)),
        ("et",ExtraTreesClassifier(n_estimators=430,max_features=0.89,min_samples_leaf=1,random_state=seed,n_jobs=-1)),
        ("mlp1",sc(MLPClassifier(hidden_layer_sizes=(128,64),alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed))),
        ("mlp2",sc(MLPClassifier(hidden_layer_sizes=(200,100),activation="tanh",alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed)))],
        voting="soft",n_jobs=-1)
cv=cross_val_score(vote(42),Xtr_i,y,cv=StratifiedKFold(5,shuffle=True,random_state=42),scoring="f1_macro",n_jobs=-1).mean()
print(f"CV macro-F1 (seed42) = {cv:.4f}  (обычная in-pipeline II = 0.8432)")

CV macro-F1 (seed42) = 0.8433  (обычная in-pipeline II = 0.8432)


**🔎** ~0.843 — то же качество, без деградации. Трансдукция помогает именно тесту, не train-CV.

In [4]:
probs=np.zeros((len(Xte_i),4))
for s in [0,1,2,3,42]:
    mdl=vote(s); mdl.fit(Xtr_i,y); probs+=mdl.predict_proba(Xte_i)
pred=(probs/5).argmax(1)
pd.DataFrame({"id":te.id,"sleep_stage":pred}).to_csv("submission_07.csv",index=False)
print("submission_07.csv записан; формат OK=",list(pd.read_csv('submission_07.csv').columns)==["id","sleep_stage"] and len(pred)==len(ss))
print("распределение:",[f'{names[i]}={(pred==i).mean()*100:.1f}%' for i in range(4)])

submission_07.csv записан; формат OK= True
распределение: ['Wake=22.6%', 'Light=25.5%', 'Deep=25.7%', 'REM=26.2%']


**🔎** Воспроизводит `submission_07.csv` (seed-bag 5 сидов, трансдуктивная импутация). Кандидат на private — другой природы импутации, чем `_05/_06`.